# Módulo 00 · Fundamentos de Python para datos — Soluciones de los ejercicios

Este notebook resuelve los 5 ejercicios del final de `notebook.ipynb` (caso *TiendaSur*).
Es **autocontenido**: primero reproduce la generación y la limpieza de los datos exactamente igual que el notebook
original (misma semilla, mismos resultados) y después resuelve cada ejercicio con código, resultado e interpretación.

> Recuerda que la consigna pide, para cada ejercicio, **una frase de conclusión para Dirección**. La encontrarás al final de cada bloque.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.precision", 2)

## Preparación: datos de TiendaSur (idéntico al notebook, secciones 3 y 4)

In [ ]:
rng = np.random.default_rng(42)

tiendas = pd.DataFrame({
    "tienda_id": [1, 2, 3, 4, 5],
    "tienda": ["Centro", "Norte", "Sur", "Puerto", "Aeropuerto"],
    "ciudad": ["Sevilla", "Sevilla", "Cádiz", "Cádiz", "Málaga"],
    "m2": [450, 300, 380, 220, 150],
})

categorias = {"Alimentación": 8, "Bebidas": 5, "Hogar": 15, "Electrónica": 60, "Ropa": 25}
fechas = pd.date_range("2025-01-01", "2025-12-31", freq="D")

n = 20_000
ventas = pd.DataFrame({
    "fecha": rng.choice(fechas, n),
    "tienda_id": rng.choice(tiendas["tienda_id"], n, p=[0.3, 0.2, 0.2, 0.15, 0.15]),
    "categoria": rng.choice(list(categorias), n, p=[0.35, 0.25, 0.15, 0.1, 0.15]),
    "promocion": rng.random(n) < 0.2,
})
precio_base = ventas["categoria"].map(categorias)
estacionalidad = 1 + 0.3 * np.sin(2 * np.pi * (ventas["fecha"].dt.dayofyear - 80) / 365)
ventas["unidades"] = rng.poisson(2 * estacionalidad * np.where(ventas["promocion"], 1.4, 1.0)) + 1
ventas["precio_unitario"] = (precio_base * rng.normal(1, 0.1, n) * np.where(ventas["promocion"], 0.85, 1)).round(2)

ventas.loc[rng.choice(n, 150, replace=False), "precio_unitario"] = np.nan
ventas = pd.concat([ventas, ventas.sample(80, random_state=1)])

# Limpieza (sección 4 del notebook)
ventas = ventas.drop_duplicates()
ventas["precio_unitario"] = ventas["precio_unitario"].fillna(
    ventas.groupby("categoria")["precio_unitario"].transform("median")
)
ventas["ingreso"] = ventas["unidades"] * ventas["precio_unitario"]
ventas["mes"] = ventas["fecha"].dt.to_period("M")

# Tabla de ventas enriquecida con los datos de cada tienda (la usaremos en varios ejercicios)
ventas_t = ventas.merge(tiendas, on="tienda_id", how="left")
print("Filas:", len(ventas_t), "| Nulos:", ventas_t.isna().sum().sum())
ventas_t.head()

## Ejercicio 1 · ¿Qué día de la semana se vende más?

**Idea:** extraemos el día de la semana de la fecha y agrupamos. Ojo con dos detalles:
- `dt.day_name()` devuelve los nombres en inglés y, si ordenamos alfabéticamente, el orden no tiene sentido. Usamos
  `dt.dayofweek` (0 = lunes, …, 6 = domingo) para ordenar y un diccionario para traducir.
- "Vender más" puede medirse en **ingreso** o en **número de tickets**. Mostramos ambos y también el ticket medio,
  para distinguir si un día vende más porque hay más tickets o porque cada ticket es mayor.

In [ ]:
nombres_dias = {0: "Lunes", 1: "Martes", 2: "Miércoles", 3: "Jueves", 4: "Viernes", 5: "Sábado", 6: "Domingo"}

ventas["dia_semana"] = ventas["fecha"].dt.dayofweek.map(nombres_dias)
# (alternativa equivalente con la pista del enunciado: ventas["fecha"].dt.day_name())

por_dia = (
    ventas.groupby(ventas["fecha"].dt.dayofweek)
    .agg(ingreso=("ingreso", "sum"), tickets=("ingreso", "size"), ticket_medio=("ingreso", "mean"))
    .rename(index=nombres_dias)
)
por_dia["cuota_%"] = 100 * por_dia["ingreso"] / por_dia["ingreso"].sum()
por_dia.sort_values("ingreso", ascending=False)

In [ ]:
dia_top = por_dia["ingreso"].idxmax()
dia_min = por_dia["ingreso"].idxmin()
dif = 100 * (por_dia["ingreso"].max() / por_dia["ingreso"].min() - 1)
print(f"Día con más ingreso: {dia_top} ({por_dia.loc[dia_top, 'ingreso']:,.0f} €)")
print(f"Día con menos ingreso: {dia_min} ({por_dia.loc[dia_min, 'ingreso']:,.0f} €)")
print(f"Diferencia entre el mejor y el peor día: {dif:.1f} %")
print(f"Rango de cuotas: {por_dia['cuota_%'].min():.1f} % – {por_dia['cuota_%'].max():.1f} % (reparto uniforme = {100/7:.1f} %)")

In [ ]:
ax = por_dia["ingreso"].plot(kind="bar", color="steelblue", figsize=(8, 3.5))
ax.axhline(por_dia["ingreso"].mean(), color="grey", ls="--", label="Media diaria")
ax.set(title="Ingreso total por día de la semana (2025)", xlabel="", ylabel="€")
ax.legend()
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Interpretación.** El domingo es el día con más ingreso (≈146.800 €) y el martes el que menos (≈136.000 €): una
diferencia del 7,9 % entre extremos. Pero todas las cuotas están entre el 13,8 % y el 14,9 %, muy cerca del 14,3 % que
correspondería a un reparto perfectamente uniforme. Además, el orden no lo explica un único factor: el lunes tiene más
tickets que el domingo, pero el domingo tiene un ticket medio algo mayor (≈51 € frente a ≈50 €). Es lo esperable en
datos simulados en los que la fecha se eligió al azar, sin ningún efecto real del día de la semana: estas diferencias
son **ruido**. En datos reales esperaríamos un patrón más claro (p. ej., sábados fuertes), y por eso conviene comprobar
si una diferencia es real o casual: lo veremos en los módulos 01 y 04.

**Conclusión para Dirección:** *el domingo es el día con más ingreso, pero solo un 7,9 % por encima del peor día
(martes) y con cuotas entre el 13,8 % y el 14,9 %; no hay un día claramente mejor que justifique, por sí solo, cambiar
horarios ni plantillas.*

## Ejercicio 2 · Cuota (%) de cada categoría sobre el ingreso total de cada ciudad

**Idea:** necesitamos la ciudad, que está en la tabla `tiendas` → `merge`. Después agrupamos por ciudad y categoría y
dividimos entre el total de la ciudad. El total de la ciudad, repetido en cada fila, se obtiene con
`groupby(...).transform("sum")` (pregunta 4 de la autoevaluación).

In [ ]:
cat_ciudad = ventas_t.groupby(["ciudad", "categoria"], as_index=False)["ingreso"].sum()
cat_ciudad["total_ciudad"] = cat_ciudad.groupby("ciudad")["ingreso"].transform("sum")
cat_ciudad["cuota_%"] = 100 * cat_ciudad["ingreso"] / cat_ciudad["total_ciudad"]
cat_ciudad.head(6)

In [ ]:
# Presentación en formato ancho: categorías en filas, ciudades en columnas (cada columna suma 100 %)
cuotas = cat_ciudad.pivot(index="categoria", columns="ciudad", values="cuota_%")
cuotas = cuotas.loc[cuotas.mean(axis=1).sort_values(ascending=False).index]
print("Comprobación (cada columna debe sumar 100):", cuotas.sum().round(1).to_dict())
cuotas.round(1)

In [ ]:
# Alternativa en una línea: tabla de contingencia normalizada por columnas
alt = pd.crosstab(ventas_t["categoria"], ventas_t["ciudad"], values=ventas_t["ingreso"],
                  aggfunc="sum", normalize="columns") * 100
print("¿Coinciden ambos métodos?", np.allclose(alt.loc[cuotas.index, cuotas.columns], cuotas))

In [ ]:
ax = cuotas.T.plot(kind="barh", stacked=True, figsize=(9, 3), colormap="tab10")
ax.set(title="Mix de ingresos por categoría en cada ciudad", xlabel="% del ingreso de la ciudad", ylabel="")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", frameon=False)
plt.tight_layout()
plt.show()

**Interpretación.** Electrónica es la primera categoría en las tres ciudades (≈38 % del ingreso) pese a ser solo
~10 % de los tickets: lo explica su precio unitario (60 € de base frente a 5–25 € del resto). Le siguen Ropa (21–23 %),
Alimentación (≈17 %), Hogar (≈14 %) y Bebidas (8 %). El mix es **casi idéntico** entre ciudades: ninguna categoría
difiere más de 2 puntos de una ciudad a otra.

**Conclusión para Dirección:** *Sevilla, Cádiz y Málaga tienen prácticamente el mismo mix de ingresos (Electrónica ≈ 38 %,
Ropa ≈ 22 %); no hay una ciudad con un perfil de demanda distinto que justifique un surtido diferenciado.*

## Ejercicio 3 · ¿Qué mes tiene el mayor ingreso en *Aeropuerto*? ¿Coincide con el del resto?

In [ ]:
mensual_tienda = pd.pivot_table(ventas_t, index="mes", columns="tienda", values="ingreso", aggfunc="sum")
mejor_mes = pd.DataFrame({
    "mejor_mes": mensual_tienda.idxmax(),
    "ingreso_mejor_mes": mensual_tienda.max(),
    "peor_mes": mensual_tienda.idxmin(),
    "ingreso_peor_mes": mensual_tienda.min(),
})
mejor_mes

In [ ]:
mes_aero = mensual_tienda["Aeropuerto"].idxmax()
otros = mejor_mes.drop(index="Aeropuerto")["mejor_mes"]
print(f"Mejor mes de Aeropuerto: {mes_aero}")
print("Mejor mes del resto:", otros.astype(str).to_dict())
print("Coincide con:", [t for t, m in otros.items() if m == mes_aero] or "ninguna")

# Ranking de meses de Aeropuerto: ¿qué distancia hay entre el 1.º y el 2.º?
top3 = mensual_tienda["Aeropuerto"].sort_values(ascending=False).head(3)
print("\nTop 3 meses de Aeropuerto:\n", top3.round(0), sep="")

In [ ]:
# Para comparar tiendas de tamaño muy distinto, indexamos cada serie: 100 = media mensual de la tienda
indice = 100 * mensual_tienda / mensual_tienda.mean()
ax = indice.drop(columns="Aeropuerto").plot(figsize=(10, 4), alpha=0.45, lw=1.2)
indice["Aeropuerto"].plot(ax=ax, color="black", lw=2.5, marker="o", label="Aeropuerto")
ax.axhline(100, color="grey", ls=":")
ax.set(title="Ingreso mensual indexado (100 = media de cada tienda)", xlabel="", ylabel="Índice")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", frameon=False)
plt.tight_layout()
plt.show()

**Interpretación.** El mejor mes de Aeropuerto es **mayo de 2025** (≈14.400 €). Solo coincide con Norte; Centro y Sur
tienen su pico en julio y Puerto en agosto. Sin embargo, la diferencia es menor de lo que parece: los tres mejores meses
de Aeropuerto (mayo, agosto y abril) están separados por poco más de un 5 %, y el gráfico indexado muestra que todas las
tiendas comparten la misma forma estacional (alta de primavera a verano, baja en invierno). En la simulación la
estacionalidad es común a toda la cadena, así que el mes exacto del pico varía de una tienda a otra por puro ruido.
El gráfico indexado es la forma justa de comparar tiendas de tamaño distinto: en euros, Aeropuerto parecería "plana"
al lado de Centro.

**Conclusión para Dirección:** *el mejor mes de Aeropuerto fue mayo, a diferencia de la mayoría de tiendas (julio-agosto),
pero la diferencia con sus meses de verano es pequeña: Aeropuerto sigue la misma temporada alta que el resto y puede
planificarse con el mismo calendario de stock y personal.*

## Ejercicio 4 · Tickets grandes y promociones

**Idea:** el percentil 90 del ingreso es el umbral que solo supera el 10 % de los tickets (`quantile(0.9)`).
La columna booleana se crea con una comparación vectorizada. Para responder bien hay que comparar con la **tasa base**:
¿qué % de *todos* los tickets ocurre en promoción?

In [ ]:
p90 = ventas["ingreso"].quantile(0.9)
ventas["ticket_grande"] = ventas["ingreso"] > p90

pct_grandes = 100 * ventas["ticket_grande"].mean()
pct_promo_en_grandes = 100 * ventas.loc[ventas["ticket_grande"], "promocion"].mean()
pct_promo_global = 100 * ventas["promocion"].mean()

print(f"Percentil 90 del ingreso por ticket: {p90:.2f} €")
print(f"Tickets grandes: {ventas['ticket_grande'].sum()} ({pct_grandes:.1f} % del total)")
print(f"% de tickets grandes que ocurren en promoción: {pct_promo_en_grandes:.1f} %")
print(f"% de todos los tickets que ocurren en promoción (tasa base): {pct_promo_global:.1f} %")
print(f"Ratio: {pct_promo_en_grandes / pct_promo_global:.2f}x")

In [ ]:
# La pregunta "inversa" también es útil: ¿qué % de tickets es grande, con y sin promoción?
tabla = pd.crosstab(ventas["promocion"], ventas["ticket_grande"], normalize="index") * 100
tabla.index = ["Sin promoción", "Con promoción"]
tabla.columns = ["Normal", "Grande"]
tabla.round(1)

In [ ]:
# ¿De qué categorías son los tickets grandes? (ayuda a interpretar)
(100 * ventas.loc[ventas["ticket_grande"], "categoria"].value_counts(normalize=True)).round(1)

**Interpretación.** El percentil 90 del ingreso por ticket es 115,16 €. El **20,2 %** de los tickets grandes ocurre
en promoción, prácticamente igual que la tasa base (el 20,0 % de *todos* los tickets): ratio 1,01. Dicho al revés, la
probabilidad de que un ticket sea grande es del 10,2 % con promoción y del 10,0 % sin ella. **No hay asociación.**

¿Por qué, si la promoción sube las unidades? Porque casi 8 de cada 10 tickets grandes son de Electrónica, y ahí actúan
dos efectos opuestos: la promoción añade unidades, pero el descuento del 15 % baja el precio. Un ticket de 2 unidades de
Electrónica sin promoción (~120 €) supera el umbral; con promoción (~102 €) se queda por debajo. Los dos efectos se
compensan. Lección general: **compara siempre con la tasa base**; un "20 % de los tickets grandes son en promoción" suena
a mucho o a poco según cuántos tickets totales haya en promoción.

**Conclusión para Dirección:** *las promociones no generan más tickets grandes: el 20 % de los tickets grandes ocurre en
promoción, exactamente la misma proporción que en el total de tickets; el descuento compensa el aumento de unidades.*

## Ejercicio 5 · ¿En qué tienda abrir una sección de Electrónica más grande?

**Idea:** la pregunta es de **comparación entre tiendas**, así que el gráfico natural es de barras. Pero ¿qué medir?
- El ingreso total de Electrónica favorece siempre a la tienda más grande (Centro) y no dice nada de la eficiencia.
- Lo relevante para decidir **dónde ampliar espacio** es la productividad: ingreso de Electrónica **por m²** de tienda
  (proxy razonable, ya que no tenemos los m² de cada sección) y el **peso** de Electrónica dentro de la tienda.

Mostramos ambas métricas lado a lado, con la media de la cadena como referencia.

In [ ]:
elec = (
    ventas_t.assign(es_elec=ventas_t["categoria"].eq("Electrónica"))
    .groupby(["tienda", "m2"], as_index=False)
    .apply(lambda d: pd.Series({
        "ingreso_elec": d.loc[d["es_elec"], "ingreso"].sum(),
        "ingreso_total": d["ingreso"].sum(),
    }), include_groups=False)
)
elec["elec_m2"] = elec["ingreso_elec"] / elec["m2"]
elec["cuota_elec_%"] = 100 * elec["ingreso_elec"] / elec["ingreso_total"]
elec = elec.sort_values("elec_m2", ascending=False).reset_index(drop=True)
elec

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colores = ["darkorange" if i == 0 else "lightgrey" for i in range(len(elec))]

sns.barplot(data=elec, x="tienda", y="elec_m2", hue="tienda", palette=colores, legend=False, ax=axes[0])
axes[0].axhline(elec["ingreso_elec"].sum() / elec["m2"].sum(), color="black", ls="--", lw=1, label="Media cadena")
axes[0].set(title="Ingreso de Electrónica por m² de tienda", xlabel="", ylabel="€/m² (2025)")
axes[0].legend()
for i, v in enumerate(elec["elec_m2"]):
    axes[0].text(i, v, f"{v:.0f}", ha="center", va="bottom")

sns.barplot(data=elec, x="tienda", y="cuota_elec_%", hue="tienda", palette=colores, legend=False, ax=axes[1])
axes[1].set(title="Peso de Electrónica en el ingreso de la tienda", xlabel="", ylabel="%")
for i, v in enumerate(elec["cuota_elec_%"]):
    axes[1].text(i, v, f"{v:.1f} %", ha="center", va="bottom")

plt.suptitle("¿Dónde ampliar Electrónica? Productividad por m² (izq.) y peso de la categoría (dcha.)", y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
top = elec.iloc[0]
media_cadena = elec["ingreso_elec"].sum() / elec["m2"].sum()
print(f"Tienda con más Electrónica por m²: {top['tienda']} ({top['elec_m2']:.0f} €/m², "
      f"{top['elec_m2'] / media_cadena:.1f} veces la media de la cadena, {media_cadena:.0f} €/m²)")

**Interpretación y justificación del gráfico.** Elegimos barras porque comparamos una magnitud entre 5 categorías
(tiendas), ordenadas de mayor a menor y con la ganadora resaltada en color, de forma que la respuesta se lee en un
segundo. Usamos € por m² y no € totales porque la decisión es de **espacio**: Centro vende más Electrónica en euros
solo porque es la tienda más grande. Aeropuerto genera 365 € de Electrónica por m², 1,5 veces la media de la cadena
(251 €/m²) y muy por delante de Puerto (272 €/m²); el peso de Electrónica dentro de cada tienda (panel derecho) es
similar en todas (37–40 %), así que la ventaja viene de la
densidad de ventas, no de un cliente distinto.

Limitaciones a comentar con Dirección: (1) no conocemos los m² que ocupa hoy cada sección ni el coste del espacio;
(2) una tienda pequeña con alta densidad puede estar ya saturada (¿hay espacio físico para ampliar?); (3) los datos son
de un único año.

**Conclusión para Dirección:** *Aeropuerto es la tienda que más ingreso de Electrónica genera por metro cuadrado
(365 €/m², 1,5 veces la media de la cadena) y es la candidata natural para ampliar la sección, siempre que haya espacio físico
disponible; Centro vende más en total, pero solo por su tamaño.*